# IN22 Alignment Benchmark for Indic Embedding Models

This notebook evaluates embedding alignment using **IN22-Gen** and **IN22-Conv** instead of Samanantar/BPCC.

It runs two benchmark types:

1. **Indic–Indic alignment**: every scheduled Indic language is compared against every other scheduled Indic language.
2. **English–Indic bidirectional alignment**: English → Indic and Indic → English.

Main metrics for alignment:

- `mean_gold_cosine`: average cosine similarity of true translation pairs
- `cosine_gap`: mean gold-pair cosine minus mean random wrong-pair cosine

Retrieval metrics are also included as supporting evidence:

- `accuracy_at_1`
- `recall_at_10`
- `mrr`

**Important:** IN22 datasets require accepting the Hugging Face dataset terms. If loading fails, open the IN22-Gen and IN22-Conv pages on Hugging Face, accept access conditions, then rerun with your HF token.


## 0. Colab setup

Use GPU: **Runtime → Change runtime type → GPU**.

Run this cell once per fresh Colab session.


In [ ]:
from pathlib import Path
import sys

for _candidate in (Path.cwd(), *Path.cwd().parents):
    _guard_dir = _candidate / "scripts"
    if (_guard_dir / "import_guard.py").exists():
        if str(_guard_dir) not in sys.path:
            sys.path.insert(0, str(_guard_dir))
        break
else:
    raise RuntimeError("Could not locate scripts/import_guard.py. Run this notebook from the WSAI workspace or copy the guard module alongside it.")

from import_guard import install_pandas_guards
install_pandas_guards()


In [ ]:
# Clean packages that can break transformers import in Colab for text-only work.
!pip -q uninstall -y torchvision torchaudio torchtext fastai timm -q

# Install text/NLP stack. Do NOT manually install torch; Colab provides GPU torch.
!pip -q install \
  "numpy==2.0.2" \
  "scipy==1.15.3" \
  "scikit-learn==1.6.1" \
  "transformers==4.48.3" \
  "sentence-transformers==3.4.1" \
  "datasets==3.2.0" \
  "accelerate==1.3.0" \
  "pandas==2.2.2" \
  "tqdm==4.67.1" \
  "matplotlib==3.10.0" \
  "pyyaml==6.0.2" \
  "sentencepiece==0.2.0" \
  "huggingface_hub>=0.27.0"

## 1. Imports, Drive, and optional Hugging Face login

In [ ]:
import os
import gc
import random
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

from datasets import load_dataset, DatasetDict

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

# Optional: login if IN22 access requires authentication.
# 1. Accept terms on HF dataset pages for ai4bharat/IN22-Gen and ai4bharat/IN22-Conv.
# 2. Paste token here OR use Colab Secrets.
HF_TOKEN = ""  # paste token only if needed
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print("Logged in to Hugging Face")
else:
    print("HF_TOKEN not set. If dataset loading fails, accept HF conditions and add token.")

# Mount Drive to save outputs permanently.
from google.colab import drive
USE_DRIVE = True

if USE_DRIVE:
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/in22_embedding_alignment_benchmark')
else:
    BASE_DIR = Path('/content/in22_embedding_alignment_benchmark')

OUTPUT_DIR = BASE_DIR / 'outputs'
(OUTPUT_DIR / 'embeddings').mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'metrics').mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'plots').mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'errors').mkdir(parents=True, exist_ok=True)

print("Output dir:", OUTPUT_DIR)

## 2. Configuration

The default language list uses **one canonical script per scheduled language** to keep the main benchmark as 22 languages.

If you want script-level evaluation too, set `INCLUDE_ALT_SCRIPTS = True`. That will add Kashmiri Devanagari, Manipuri Bengali, and Sindhi Devanagari as separate language-script variants.


In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

# Full evaluation: QUICK_N = 0
# Dry run: set QUICK_N = 200 or 300 first.
QUICK_N = 0
BATCH_SIZE = 32
MAX_LENGTH = 128

# The two IN22 suites to evaluate.
IN22_DATASETS = {
    'IN22-Gen': 'ai4bharat/IN22-Gen',
    'IN22-Conv': 'ai4bharat/IN22-Conv',
}

# Main 22 scheduled Indic languages: one script per language.
# Note: IN22 also has alternate scripts for Kashmiri, Manipuri/Meitei, and Sindhi.
INDIC_22 = {
    'asm': 'asm_Beng',  # Assamese
    'ben': 'ben_Beng',  # Bengali
    'brx': 'brx_Deva',  # Bodo
    'doi': 'doi_Deva',  # Dogri
    'guj': 'guj_Gujr',  # Gujarati
    'hin': 'hin_Deva',  # Hindi
    'kan': 'kan_Knda',  # Kannada
    'kas': 'kas_Arab',  # Kashmiri, Arabic script selected as canonical here
    'gom': 'gom_Deva',  # Konkani
    'mai': 'mai_Deva',  # Maithili
    'mal': 'mal_Mlym',  # Malayalam
    'mni': 'mni_Mtei',  # Manipuri/Meitei, Meitei script selected as canonical here
    'mar': 'mar_Deva',  # Marathi
    'npi': 'npi_Deva',  # Nepali
    'ory': 'ory_Orya',  # Odia
    'pan': 'pan_Guru',  # Punjabi
    'san': 'san_Deva',  # Sanskrit
    'sat': 'sat_Olck',  # Santali
    'snd': 'snd_Arab',  # Sindhi, Arabic script selected as canonical here
    'tam': 'tam_Taml',  # Tamil
    'tel': 'tel_Telu',  # Telugu
    'urd': 'urd_Arab',  # Urdu
}

ALT_SCRIPT_VARIANTS = {
    'kas_deva': 'kas_Deva',  # Kashmiri Devanagari
    'mni_beng': 'mni_Beng',  # Manipuri Bengali script
    'snd_deva': 'snd_Deva',  # Sindhi Devanagari
}

INCLUDE_ALT_SCRIPTS = False

LANGS = {'eng': 'eng_Latn', **INDIC_22}
if INCLUDE_ALT_SCRIPTS:
    LANGS.update(ALT_SCRIPT_VARIANTS)

INDIC_KEYS = list(INDIC_22.keys())
if INCLUDE_ALT_SCRIPTS:
    INDIC_KEYS = list(INDIC_22.keys()) + list(ALT_SCRIPT_VARIANTS.keys())

print('Number of Indic language entries:', len(INDIC_KEYS))
print('Indic keys:', INDIC_KEYS)
print('Total language entries including English:', len(LANGS))

# Models discussed in this work.
MODELS = [
    {'name': 'labse', 'hf_id': 'sentence-transformers/LaBSE', 'kind': 'sentence_transformer'},
    {'name': 'mpnet_multilingual', 'hf_id': 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2', 'kind': 'sentence_transformer'},
    {'name': 'vyakyarth', 'hf_id': 'krutrim-ai-labs/Vyakyarth', 'kind': 'sentence_transformer'},
    {'name': 'sbert_multilingual_minilm', 'hf_id': 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2', 'kind': 'sentence_transformer'},
    {'name': 'muril', 'hf_id': 'google/muril-base-cased', 'kind': 'hf_mean_pool'},
    {'name': 'indicbertv2_ss', 'hf_id': 'ai4bharat/IndicBERTv2-SS', 'kind': 'hf_mean_pool', 'trust_remote_code': True},
    {'name': 'xlm_roberta_base', 'hf_id': 'FacebookAI/xlm-roberta-base', 'kind': 'hf_mean_pool'},
    {'name': 'simcse_xlmr_multilingual', 'hf_id': 'ZurichNLP/unsup-simcse-xlm-roberta-base', 'kind': 'hf_mean_pool'},
    {'name': 'mcontriever', 'hf_id': 'facebook/mcontriever', 'kind': 'hf_mean_pool'},
    {'name': 'contriever', 'hf_id': 'facebook/contriever', 'kind': 'hf_mean_pool'},
    {'name': 'simcse_roberta_supervised', 'hf_id': 'princeton-nlp/sup-simcse-roberta-base', 'kind': 'hf_mean_pool'},
]

# To run only a subset during testing, edit this list.
RUN_MODEL_NAMES = [m['name'] for m in MODELS]
# Example quick test:
# RUN_MODEL_NAMES = ['labse', 'mpnet_multilingual']

MODELS_TO_RUN = [m for m in MODELS if m['name'] in RUN_MODEL_NAMES]
print('Models to run:', [m['name'] for m in MODELS_TO_RUN])

## 3. Load IN22-Gen and IN22-Conv

We load the `all` configuration because it gives aligned sentence columns such as `sentence_eng_Latn`, `sentence_hin_Deva`, etc.

The same row index across languages has the same meaning, so it is valid for both English–Indic and Indic–Indic alignment evaluation.


In [ ]:
def get_dataset_split(ds_obj):
    """Return the actual Dataset object from a DatasetDict or Dataset."""
    if isinstance(ds_obj, DatasetDict):
        if 'gen' in ds_obj:
            return ds_obj['gen']
        first_key = list(ds_obj.keys())[0]
        print('Using split:', first_key)
        return ds_obj[first_key]
    return ds_obj


def load_in22_all(dataset_label: str, dataset_id: str, langs: Dict[str, str], quick_n: int = 0) -> Dict[str, List[str]]:
    print(f"\nLoading {dataset_label}: {dataset_id}")

    try:
        ds = load_dataset(dataset_id, 'all', split='gen')
    except Exception as e:
        print('Direct split=gen load failed; trying DatasetDict load...')
        print(type(e).__name__, ':', e)
        ds_obj = load_dataset(dataset_id, 'all')
        ds = get_dataset_split(ds_obj)

    print(ds)
    print('Columns:', ds.column_names[:10], '... total:', len(ds.column_names))

    if quick_n and quick_n > 0:
        ds = ds.select(range(min(quick_n, len(ds))))

    texts_by_lang = {}
    missing_cols = []

    for short, code in langs.items():
        col = f'sentence_{code}'
        if col not in ds.column_names:
            missing_cols.append(col)
            continue
        texts = [str(x).strip() for x in ds[col]]
        texts_by_lang[short] = texts
        print(f'{short:10s} {code:10s} -> {len(texts)} sentences')

    if missing_cols:
        raise ValueError(f'Missing columns in {dataset_label}: {missing_cols}')

    lengths = {k: len(v) for k, v in texts_by_lang.items()}
    if len(set(lengths.values())) != 1:
        raise ValueError(f'Unequal language lengths: {lengths}')

    print(f'{dataset_label} loaded with {next(iter(lengths.values()))} aligned rows.')
    return texts_by_lang

texts_by_suite = {}
for suite_name, ds_id in IN22_DATASETS.items():
    texts_by_suite[suite_name] = load_in22_all(suite_name, ds_id, LANGS, QUICK_N)

# Show a sample alignment
for suite_name, texts_by_lang in texts_by_suite.items():
    print('\nSample from', suite_name)
    print('ENG:', texts_by_lang['eng'][0])
    print('HIN:', texts_by_lang['hin'][0])
    print('TAM:', texts_by_lang['tam'][0])

## 4. Embedding model helpers

In [ ]:
@dataclass
class ModelSpec:
    name: str
    hf_id: str
    kind: str
    trust_remote_code: bool = False


def mean_pool(last_hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts


class Embedder:
    def __init__(self, spec: ModelSpec, device: str, max_length: int):
        self.spec = spec
        self.device = device
        self.max_length = max_length

        if spec.kind == 'sentence_transformer':
            from sentence_transformers import SentenceTransformer
            print('Loading as SentenceTransformer:', spec.hf_id)
            self.model = SentenceTransformer(
                spec.hf_id,
                device=device,
                trust_remote_code=spec.trust_remote_code,
            )
            if device == 'cuda':
                self.model = self.model.half()
            self.tokenizer = None

        elif spec.kind == 'hf_mean_pool':
            from transformers import AutoModel, AutoTokenizer
            print('Loading as HF mean-pool model:', spec.hf_id)
            self.tokenizer = AutoTokenizer.from_pretrained(
                spec.hf_id,
                trust_remote_code=spec.trust_remote_code,
            )
            dtype = torch.float16 if device == 'cuda' else torch.float32
            self.model = AutoModel.from_pretrained(
                spec.hf_id,
                trust_remote_code=spec.trust_remote_code,
                torch_dtype=dtype,
            )
            self.model.to(device)
            self.model.eval()
        else:
            raise ValueError(f'Unknown model kind: {spec.kind}')

    @torch.no_grad()
    def encode(self, texts: List[str], batch_size: int) -> np.ndarray:
        if self.spec.kind == 'sentence_transformer':
            emb = self.model.encode(
                texts,
                batch_size=batch_size,
                convert_to_numpy=True,
                normalize_embeddings=True,
                show_progress_bar=True,
            )
            return emb.astype('float32')

        all_embeddings = []
        for start in tqdm(range(0, len(texts), batch_size), desc=f'Encoding {self.spec.name}'):
            batch = texts[start:start + batch_size]
            encoded = self.tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=self.max_length,
                return_tensors='pt',
            )
            encoded = {k: v.to(self.device) for k, v in encoded.items()}
            outputs = self.model(**encoded)
            pooled = mean_pool(outputs.last_hidden_state, encoded['attention_mask'])
            pooled = F.normalize(pooled, p=2, dim=1)
            all_embeddings.append(pooled.detach().cpu().float().numpy())

        return np.vstack(all_embeddings).astype('float32')


def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 5. Metric functions

Primary alignment metrics:

- `mean_gold_cosine`
- `cosine_gap`

Supporting retrieval metrics:

- `accuracy_at_1`
- `recall_at_10`
- `mrr`


In [ ]:
def cosine_sim_matrix(src_emb: np.ndarray, tgt_emb: np.ndarray) -> np.ndarray:
    return cosine_similarity(src_emb, tgt_emb).astype('float32')


def compute_alignment_and_retrieval_metrics(
    src_emb: np.ndarray,
    tgt_emb: np.ndarray,
    seed: int = 42,
    top_ks: Tuple[int, ...] = (1, 5, 10, 50),
) -> Dict[str, float]:

    sim = cosine_sim_matrix(src_emb, tgt_emb)
    n = sim.shape[0]

    # Gold translation pairs are diagonal because IN22 is row-aligned.
    gold = np.diag(sim)

    # Random negative pairs for cosine gap.
    rng = np.random.default_rng(seed)
    neg_idx = rng.permutation(n)
    for i in range(n):
        if neg_idx[i] == i:
            neg_idx[i] = (neg_idx[i] + 1) % n
    random_neg = sim[np.arange(n), neg_idx]

    # Retrieval ranks: where is target row i in the sorted candidate list?
    sorted_idx = np.argsort(-sim, axis=1)
    ranks = np.empty(n, dtype=np.int64)
    for i in range(n):
        ranks[i] = int(np.where(sorted_idx[i] == i)[0][0]) + 1

    out = {
        'n_examples': int(n),

        # Main alignment numbers
        'mean_gold_cosine': float(np.mean(gold)),
        'median_gold_cosine': float(np.median(gold)),
        'std_gold_cosine': float(np.std(gold)),
        'min_gold_cosine': float(np.min(gold)),
        'max_gold_cosine': float(np.max(gold)),
        'pct_gold_gt_0_50': float(np.mean(gold > 0.50)),
        'pct_gold_gt_0_60': float(np.mean(gold > 0.60)),
        'pct_gold_gt_0_70': float(np.mean(gold > 0.70)),
        'pct_gold_gt_0_80': float(np.mean(gold > 0.80)),
        'pct_gold_gt_0_90': float(np.mean(gold > 0.90)),

        # Main separation/uniformity diagnostic
        'mean_random_cosine': float(np.mean(random_neg)),
        'std_random_cosine': float(np.std(random_neg)),
        'cosine_gap': float(np.mean(gold) - np.mean(random_neg)),

        # Supporting retrieval numbers
        'mrr': float(np.mean(1.0 / ranks)),
        'mean_rank': float(np.mean(ranks)),
        'median_rank': float(np.median(ranks)),
    }

    for k in top_ks:
        out[f'recall_at_{k}'] = float(np.mean(ranks <= k))

    out['accuracy_at_1'] = out['recall_at_1']
    return out


def collect_errors(src_texts, tgt_texts, src_emb, tgt_emb, n_examples=25):
    sim = cosine_sim_matrix(src_emb, tgt_emb)
    pred_idx = np.argmax(sim, axis=1)
    rows = []
    for i, p in enumerate(pred_idx):
        if p != i:
            rows.append({
                'row_id': i,
                'source_text': src_texts[i],
                'gold_translation': tgt_texts[i],
                'predicted_neighbor': tgt_texts[p],
                'gold_cosine': float(sim[i, i]),
                'predicted_cosine': float(sim[i, p]),
                'margin_pred_minus_gold': float(sim[i, p] - sim[i, i]),
            })
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    return df.sort_values('margin_pred_minus_gold', ascending=False).head(n_examples)

## 6. Pair generation

Two benchmark types are created:

1. `indic_indic`: Indic → Indic only, excluding same-language pairs.
2. `eng_indic_bidir`: English → Indic and Indic → English.


In [ ]:
def make_pairs(benchmark_type: str) -> List[Tuple[str, str]]:
    if benchmark_type == 'indic_indic':
        return [(src, tgt) for src in INDIC_KEYS for tgt in INDIC_KEYS if src != tgt]
    if benchmark_type == 'eng_indic_bidir':
        return [('eng', tgt) for tgt in INDIC_KEYS] + [(src, 'eng') for src in INDIC_KEYS]
    raise ValueError(f'Unknown benchmark_type: {benchmark_type}')

BENCHMARK_TYPES = ['indic_indic', 'eng_indic_bidir']

for btype in BENCHMARK_TYPES:
    pairs = make_pairs(btype)
    print(btype, 'number of directed pairs:', len(pairs))
    print('first 10:', pairs[:10])

## 7. Run benchmark

This is the main cell. It encodes every language once per model and dataset suite, then evaluates all requested directions.

If you want a quick test, set `RUN_MODEL_NAMES = ['labse']` and `QUICK_N = 200` above.


In [ ]:
# Tags for cache filenames
N_TAG = QUICK_N if QUICK_N and QUICK_N > 0 else 'full'
LANG_TAG = '22sched' if not INCLUDE_ALT_SCRIPTS else '25script'

SAVE_ERRORS = False  # set True only for small runs; full run creates many files

all_rows = []

for suite_name, texts_by_lang in texts_by_suite.items():
    print('\n' + '=' * 100)
    print('DATASET SUITE:', suite_name)
    print('=' * 100)

    for model_dict in MODELS_TO_RUN:
        spec = ModelSpec(**model_dict)
        print(f"\n===== Model: {spec.name} | {spec.hf_id} | suite={suite_name} =====")

        embedder = None
        try:
            embedder = Embedder(spec, DEVICE, MAX_LENGTH)
            emb_by_lang = {}

            # Encode every language needed once.
            for lang, texts in texts_by_lang.items():
                cache_path = (
                    OUTPUT_DIR / 'embeddings' /
                    f"{spec.name}_{suite_name}_{LANG_TAG}_{lang}_n{N_TAG}_l{MAX_LENGTH}.npy"
                )

                if cache_path.exists():
                    print('Loading cached:', cache_path.name)
                    emb = np.load(cache_path)
                    if emb.shape[0] != len(texts):
                        print(f'Cache mismatch. Expected {len(texts)}, got {emb.shape[0]}. Recomputing...')
                        emb = embedder.encode(texts, batch_size=BATCH_SIZE)
                        np.save(cache_path, emb)
                        print('Saved corrected cache:', cache_path.name)
                else:
                    emb = embedder.encode(texts, batch_size=BATCH_SIZE)
                    np.save(cache_path, emb)
                    print('Saved:', cache_path.name)

                emb_by_lang[lang] = emb
                print(f'Embedding stored | suite={suite_name} | model={spec.name} | lang={lang} | shape={emb.shape}')

            # Evaluate both benchmark types.
            for benchmark_type in BENCHMARK_TYPES:
                pairs = make_pairs(benchmark_type)
                print(f'\nRunning benchmark_type={benchmark_type}, pairs={len(pairs)}')

                for src_lang, tgt_lang in tqdm(pairs, desc=f'{spec.name} {suite_name} {benchmark_type}'):
                    src_emb = emb_by_lang[src_lang]
                    tgt_emb = emb_by_lang[tgt_lang]

                    if src_emb.shape[0] != tgt_emb.shape[0]:
                        raise ValueError(f'Length mismatch: {src_lang}->{tgt_lang}: {src_emb.shape[0]} vs {tgt_emb.shape[0]}')

                    metrics = compute_alignment_and_retrieval_metrics(src_emb, tgt_emb, seed=SEED)

                    row = {
                        'dataset_suite': suite_name,
                        'benchmark_type': benchmark_type,
                        'model': spec.name,
                        'hf_id': spec.hf_id,
                        'source_language': src_lang,
                        'target_language': tgt_lang,
                        'source_code': LANGS[src_lang],
                        'target_code': LANGS[tgt_lang],
                        'language_pair': f'{src_lang}-{tgt_lang}',
                        'language_tag': LANG_TAG,
                        **metrics,
                    }
                    all_rows.append(row)

                    if SAVE_ERRORS:
                        errors = collect_errors(
                            texts_by_lang[src_lang],
                            texts_by_lang[tgt_lang],
                            src_emb,
                            tgt_emb,
                            n_examples=25,
                        )
                        error_path = (
                            OUTPUT_DIR / 'errors' /
                            f"{spec.name}_{suite_name}_{benchmark_type}_{src_lang}-{tgt_lang}_{LANG_TAG}_n{N_TAG}.csv"
                        )
                        errors.to_csv(error_path, index=False)

            # Save partial metrics after every model.
            partial_df = pd.DataFrame(all_rows)
            partial_path = OUTPUT_DIR / 'metrics' / f'in22_alignment_metrics_PARTIAL_{LANG_TAG}_n{N_TAG}.csv'
            partial_df.to_csv(partial_path, index=False)
            print('Saved partial metrics:', partial_path, 'shape:', partial_df.shape)

        except Exception as e:
            print('FAILED model:', spec.name, 'on suite:', suite_name)
            print(type(e).__name__, ':', e)
            print('Skipping this model/suite and continuing.')

        finally:
            if embedder is not None:
                del embedder
            clear_memory()

metrics_df = pd.DataFrame(all_rows)
metrics_path = OUTPUT_DIR / 'metrics' / f'in22_alignment_metrics_ALL_MODELS_{LANG_TAG}_n{N_TAG}.csv'
metrics_df.to_csv(metrics_path, index=False)
print('\nSaved full metrics:', metrics_path)
print('metrics_df shape:', metrics_df.shape)
display(metrics_df.head())

## 8. Summary tables

Primary summary is by:

- model
- dataset suite
- benchmark type

Main columns to read first:

- `mean_gold_cosine`
- `cosine_gap`
- `accuracy_at_1`


In [ ]:
if metrics_df.empty:
    raise ValueError('metrics_df is empty. Run the benchmark cell first.')

summary = (
    metrics_df
    .groupby(['model', 'dataset_suite', 'benchmark_type'])[
        ['mean_gold_cosine', 'median_gold_cosine', 'pct_gold_gt_0_80',
         'mean_random_cosine', 'cosine_gap', 'accuracy_at_1', 'recall_at_10', 'mrr']
    ]
    .mean()
    .reset_index()
    .sort_values(['benchmark_type', 'dataset_suite', 'cosine_gap'], ascending=[True, True, False])
)

summary_path = OUTPUT_DIR / 'metrics' / f'in22_model_summary_by_suite_and_benchmark_{LANG_TAG}_n{N_TAG}.csv'
summary.to_csv(summary_path, index=False)
print('Saved:', summary_path)
display(summary)

# Overall average across IN22-Gen and IN22-Conv for each benchmark type.
overall_summary = (
    metrics_df
    .groupby(['model', 'benchmark_type'])[
        ['mean_gold_cosine', 'median_gold_cosine', 'pct_gold_gt_0_80',
         'mean_random_cosine', 'cosine_gap', 'accuracy_at_1', 'recall_at_10', 'mrr']
    ]
    .mean()
    .reset_index()
    .sort_values(['benchmark_type', 'cosine_gap'], ascending=[True, False])
)

overall_path = OUTPUT_DIR / 'metrics' / f'in22_overall_model_summary_{LANG_TAG}_n{N_TAG}.csv'
overall_summary.to_csv(overall_path, index=False)
print('Saved:', overall_path)
display(overall_summary)

## 9. Plots

Since the main goal is **alignment**, the first two plots are:

1. Mean gold-pair cosine by model
2. Cosine gap by model

Accuracy@1 is plotted as a supporting retrieval/uniformity indicator.


In [ ]:
def plot_metric_by_model(df, metric, title, filename):
    plot_df = (
        df.groupby(['model', 'benchmark_type'])[metric]
        .mean()
        .reset_index()
        .pivot(index='model', columns='benchmark_type', values=metric)
    )
    # Sort by Indic-Indic value if present; otherwise by first column.
    sort_col = 'indic_indic' if 'indic_indic' in plot_df.columns else plot_df.columns[0]
    plot_df = plot_df.sort_values(sort_col, ascending=False)

    ax = plot_df.plot(kind='bar', figsize=(12, 5))
    ax.set_ylabel(metric)
    ax.set_title(title)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    path = OUTPUT_DIR / 'plots' / filename
    plt.savefig(path, dpi=200)
    plt.show()
    print('Saved plot:', path)

plot_metric_by_model(
    metrics_df,
    'mean_gold_cosine',
    'IN22 mean gold-pair cosine by model',
    f'in22_mean_gold_cosine_by_model_{LANG_TAG}_n{N_TAG}.png'
)

plot_metric_by_model(
    metrics_df,
    'cosine_gap',
    'IN22 cosine gap by model',
    f'in22_cosine_gap_by_model_{LANG_TAG}_n{N_TAG}.png'
)

plot_metric_by_model(
    metrics_df,
    'accuracy_at_1',
    'IN22 Accuracy@1 by model',
    f'in22_accuracy_at_1_by_model_{LANG_TAG}_n{N_TAG}.png'
)

## 10. Heatmap for one model

Use this to inspect model performance pair-by-pair.


In [ ]:
def plot_heatmap_for_model(df, model_name, dataset_suite='IN22-Gen', benchmark_type='indic_indic', metric='cosine_gap'):
    sub = df[
        (df['model'] == model_name) &
        (df['dataset_suite'] == dataset_suite) &
        (df['benchmark_type'] == benchmark_type)
    ].copy()

    if sub.empty:
        print('No rows found for this selection.')
        return

    heat = sub.pivot(index='source_language', columns='target_language', values=metric)

    plt.figure(figsize=(11, 9))
    plt.imshow(heat, aspect='auto')
    plt.colorbar(label=metric)
    plt.xticks(range(len(heat.columns)), heat.columns, rotation=90)
    plt.yticks(range(len(heat.index)), heat.index)
    plt.title(f'{model_name} | {dataset_suite} | {benchmark_type} | {metric}')
    plt.xlabel('Target language')
    plt.ylabel('Source language')
    plt.tight_layout()

    path = OUTPUT_DIR / 'plots' / f'heatmap_{model_name}_{dataset_suite}_{benchmark_type}_{metric}_{LANG_TAG}_n{N_TAG}.png'
    plt.savefig(path, dpi=200)
    plt.show()
    print('Saved heatmap:', path)

# Example heatmaps for LaBSE.
plot_heatmap_for_model(metrics_df, 'labse', dataset_suite='IN22-Gen', benchmark_type='indic_indic', metric='mean_gold_cosine')
plot_heatmap_for_model(metrics_df, 'labse', dataset_suite='IN22-Gen', benchmark_type='indic_indic', metric='cosine_gap')

## 11. Best and weakest Indic–Indic directions

This helps find which Indic–Indic pairs are strongest/weakest for each model.


In [ ]:
indic_indic_df = metrics_df[metrics_df['benchmark_type'] == 'indic_indic'].copy()

best_pairs = (
    indic_indic_df
    .sort_values(['model', 'cosine_gap'], ascending=[True, False])
    .groupby('model')
    .head(10)
)

weakest_pairs = (
    indic_indic_df
    .sort_values(['model', 'cosine_gap'], ascending=[True, True])
    .groupby('model')
    .head(10)
)

best_path = OUTPUT_DIR / 'metrics' / f'in22_best_indic_indic_pairs_by_cosine_gap_{LANG_TAG}_n{N_TAG}.csv'
weak_path = OUTPUT_DIR / 'metrics' / f'in22_weakest_indic_indic_pairs_by_cosine_gap_{LANG_TAG}_n{N_TAG}.csv'
best_pairs.to_csv(best_path, index=False)
weakest_pairs.to_csv(weak_path, index=False)

print('Saved:', best_path)
display(best_pairs[['dataset_suite', 'model', 'language_pair', 'mean_gold_cosine', 'mean_random_cosine', 'cosine_gap', 'accuracy_at_1', 'mrr']])

print('Saved:', weak_path)
display(weakest_pairs[['dataset_suite', 'model', 'language_pair', 'mean_gold_cosine', 'mean_random_cosine', 'cosine_gap', 'accuracy_at_1', 'mrr']])

## 12. Final interpretation guide

Use this logic in your report:

- **High mean gold cosine + high cosine gap**: strong alignment and good separation.
- **High mean gold cosine + low cosine gap**: correct translations are close, but wrong sentences may also be close. This indicates weak uniformity/separation.
- **Low mean gold cosine + low cosine gap**: weak alignment.
- **High Accuracy@1** supports that the model is practically useful for nearest-neighbour translation retrieval.

For the mentor’s current question, prioritize **Indic–Indic `mean_gold_cosine` and `cosine_gap`** first, then use Accuracy@1/MRR as supporting evidence.
